# Batch Motion-Compensation TIC Pipeline (MedSAM2)

Batch version of the single-case `MedSam2_inference` notebook.

For every row in the tracking CSV it will:
1. Auto-locate the **BMODE**, **CEUS** and **MC_VOI** files in the row's `Data Dir`
   using the `Site-Patient-Visit-Bolus` naming convention.
2. Run the same MedSAM2 / MC / no-MC TIC pipeline.
3. Create a per-case output folder `{Site}-{Patient}-{Visit}-{Bolus}/` under the
   destination directory and write the raw + fitted TIC curves there.
4. Append the fitted lognormal parameters and quality metrics back into the master CSV.

> **Inputs per case:** `*-BMODE.nii`, `*-CEUS.nii`, `*-MC_VOI.nii.gz`
> **Master CSV:** `/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/Motion Compensation Comparison 4 patients(MedSAM2).csv`

## 0. Setup & working directory
Same `os.chdir` step as the original notebook so the `src.*` imports resolve.

In [6]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print("Start CWD:", Path().cwd())
# Original notebook lives in a sub-folder; step up one level so `src` is importable.
# Adjust this if you launch the batch notebook from a different location.
if Path("src").exists() is False and Path("../src").exists():
    os.chdir(Path(os.getcwd()).parent)
print("Working CWD:", Path().cwd())

Start CWD: /home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus
Working CWD: /home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus


## 1. Configuration

Edit the paths here. `MASTER_CSV` is both read (to know which cases to process)
and written (results are appended to the matching row).

In [7]:
# ── Master tracking CSV (read cases from here, write results back here) ────────
MASTER_CSV = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/Motion Compensation Comparison 4 patients(MedSAM2).csv"

# ── Destination root for per-case output folders ──────────────────────────────
# Falls back to each row's "TIC curve save path" column when present; otherwise
# this default is used. Per-case files are overwritten in place.
DEST_ROOT = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results"

# ── MedSAM2 model ─────────────────────────────────────────────────────────────
MEDSAM2_PATH = os.path.abspath(os.path.join(os.getcwd(), "..","..", "MedSAM2"))
# The 3DMPUS fine-tuned checkpoint, not the stock MedSAM2_MRI_LiverLesion.pt.
MEDSAM2_CHECKPOINT = os.path.join(MEDSAM2_PATH, "exp_log", "3DMPUS_liver_tumor",
                                  "checkpoints", "checkpoint.pt")
MODEL_CFG = "configs/sam2.1_hiera_t512.yaml"
DEVICE    = "cuda"

# ── Adaptive-bbox segmentation options ────────────────────────────────────────
# Passed straight to Medsam2AdaptiveBboxMasker, so this batch run and the 2D
# evaluation in MedSam2_fineTuning_extract measure the same thing.
SMOOTH_SIGMA_MM   = 1.0    # 3D Gaussian over the reconstructed mask; 0 disables
SYNTHETIC_CORONAL = False  # True = model X(z) from centre-axial + sagittal instead
                           #        of segmenting a coronal guide slice
BBOX_PADDING      = 4      # per-side padding on each axial slice's adaptive bbox

# ── Loaders (match the original notebook) ─────────────────────────────────────
SCAN_TYPE = "nifti"
SEG_TYPE  = "nifti"

# Set to a list of case keys e.g. ["UCSD-P05-V01-CE1"] to only run those,
# or leave as None to process every row in the CSV.
ONLY_CASES = None

# Regenerating the MedSAM2 analysis is the point of this run, so finished rows
# are NOT skipped -- set False to resume a partial batch instead.
OVERWRITE = True

## 2. Imports & adaptive-bbox masker

The MedSAM2 segmentation is **not** reimplemented here. It comes from
`medsam2_function/medsam2_3d_mask.Medsam2AdaptiveBboxMasker`, the same class the
interactive viewer and the 2D evaluation use, so a TIC computed in this batch and
a Dice measured in `MedSam2_fineTuning_extract` describe the same masks.

Earlier versions of this notebook carried their own copies of `run_medsam2_2d`,
`get_adaptive_bbox_at_z` and the per-frame 3D reconstruction. Those copies had
drifted from the originals; they are gone.

In [8]:
import sys
import re
import glob
import time
import traceback
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # batch: render to file, no GUI windows
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import torch

# Make MedSAM2 and the shared medsam2_function helpers importable
print("MedSAM2 path:", MEDSAM2_PATH)
for _p in (MEDSAM2_PATH,
           os.path.join(os.getcwd(), "CLI-Demos"),
           os.path.join(os.getcwd(), "CLI-Demos", "medsam2_function")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Project entrypoints (same as single-case notebook)
from src.entrypoints import scan_loading_step, seg_loading_step

# The canonical adaptive-bbox implementation — imported, not copied.
from medsam2_3d_mask import Medsam2AdaptiveBboxMasker

assert os.path.isfile(MEDSAM2_CHECKPOINT), f"checkpoint not found: {MEDSAM2_CHECKPOINT}"
print("checkpoint:", MEDSAM2_CHECKPOINT)
print(f"adaptive-bbox options: smooth_sigma_mm={SMOOTH_SIGMA_MM}, "
      f"synthetic_coronal={SYNTHETIC_CORONAL}, padding={BBOX_PADDING}")

MedSAM2 path: /home/ahmed-el-kaffas/Documents/Github/QuantUS/MedSAM2
checkpoint: /home/ahmed-el-kaffas/Documents/Github/QuantUS/MedSAM2/exp_log/3DMPUS_liver_tumor/checkpoints/checkpoint.pt
adaptive-bbox options: smooth_sigma_mm=1.0, synthetic_coronal=False, padding=4


## 3. File discovery

Given a row's `Data Dir`, `Site`, `Patient`, `Visit`, `Bolus`, locate the three input
files. The convention from the screenshot is:

```
{Site}-{Patient}-{Visit}-{Bolus}_<timestamp>_..._BMODE.nii
{Site}-{Patient}-{Visit}-{Bolus}_<timestamp>_..._CEUS.nii
{Site}-{Patient}-{Visit}-{Bolus}-MC_VOI.nii.gz
```

Matching is done case-insensitively and tolerates the trailing-space / `P010`-vs-`P10`
quirks present in the tracking CSV by globbing on the `Site-Patient-Visit-Bolus` stem
and falling back to looser patterns.

In [9]:
def _clean(s):
    """Trim whitespace; keep original casing for building the case key."""
    return str(s).strip()

def make_case_key(site, patient, visit, bolus):
    return f"{_clean(site)}-{_clean(patient)}-{_clean(visit)}-{_clean(bolus)}"

def _find_one(data_dir, stem, suffix_patterns, label):
    """
    Search data_dir (recursively) for a file whose name starts with `stem`
    (case-insensitive) and matches one of the suffix patterns.
    suffix_patterns: list of glob-style endings, tried in order.
    Returns the first match or None.
    """
    data_dir = _clean(data_dir)
    candidates = []
    # gather all files once (recursive so V01 sub-folders etc. are covered)
    all_files = glob.glob(os.path.join(data_dir, "**", "*"), recursive=True)
    stem_lower = stem.lower()
    for pat in suffix_patterns:
        pat_lower = pat.lower()
        for fp in all_files:
            name = os.path.basename(fp).lower()
            if name.startswith(stem_lower) and name.endswith(pat_lower):
                candidates.append(fp)
        if candidates:
            break
    if not candidates:
        return None
    # deterministic: shortest path / first sorted
    candidates = sorted(set(candidates))
    if len(candidates) > 1:
        print(f"    [warn] multiple {label} matches for stem '{stem}', using first:")
        for c in candidates:
            print("        -", os.path.basename(c))
    return candidates[0]

def locate_inputs(row):
    """
    Resolve BMODE / CEUS / MC_VOI absolute paths for a CSV row.
    Returns dict with keys bmode, ceus, seg (any may be None if not found).
    """
    site    = _clean(row["Site"])
    patient = _clean(row["Patient Number"])
    visit   = _clean(row["Visit"])
    bolus   = _clean(row["Bolus"])
    data_dir = _clean(row["Data Dir"])

    stem = f"{site}-{patient}-{visit}-{bolus}"   # e.g. UCSD-P05-V01-CE1

    bmode = _find_one(data_dir, stem, ["_BMODE.nii", "_bmode.nii", "BMODE.nii"], "BMODE")
    ceus  = _find_one(data_dir, stem, ["_CEUS.nii", "_ceus.nii", "CEUS.nii"], "CEUS")
    # MC_VOI: stem-MC_VOI.nii.gz
    seg   = _find_one(data_dir, stem, ["-MC_VOI.nii.gz", "-mc_voi.nii.gz", "MC_VOI.nii.gz"], "MC_VOI")

    # Fallback: if the strict stem failed (e.g. P010 dir is actually P10),
    # retry with a looser stem that drops the patient/visit prefix.
    if bmode is None or ceus is None or seg is None:
        loose = f"{site}-{patient}".replace(" ", "")
        if bmode is None:
            bmode = _find_one(data_dir, "", [f"*{visit}*{bolus}*_BMODE.nii"], "BMODE(loose)")
        if ceus is None:
            ceus = _find_one(data_dir, "", [f"*{visit}*{bolus}*_CEUS.nii"], "CEUS(loose)")
        if seg is None:
            seg = _find_one(data_dir, "", [f"*{visit}*{bolus}*MC_VOI.nii.gz"], "MC_VOI(loose)")

    return {"bmode": bmode, "ceus": ceus, "seg": seg, "stem": stem}

> **Note on the loose fallback:** the `*` patterns above are matched with `fnmatch`,
> so the next cell installs that behaviour into `_find_one`.

In [10]:
import fnmatch

def _find_one(data_dir, stem, suffix_patterns, label):
    """
    Locate a file under data_dir (recursive).
    - If `stem` is non-empty: name must start with stem AND end with one of suffix_patterns.
    - If `stem` is empty: each pattern is treated as a full fnmatch glob on the basename.
    Case-insensitive throughout.
    """
    data_dir = _clean(data_dir)
    if not os.path.isdir(data_dir):
        print(f"    [warn] data dir does not exist: {data_dir}")
        return None
    all_files = glob.glob(os.path.join(data_dir, "**", "*"), recursive=True)
    stem_lower = stem.lower()
    candidates = []
    for pat in suffix_patterns:
        pat_lower = pat.lower()
        for fp in all_files:
            if not os.path.isfile(fp):
                continue
            name = os.path.basename(fp).lower()
            if stem:
                if name.startswith(stem_lower) and name.endswith(pat_lower):
                    candidates.append(fp)
            else:
                if fnmatch.fnmatch(name, pat_lower):
                    candidates.append(fp)
        if candidates:
            break
    if not candidates:
        return None
    candidates = sorted(set(candidates))
    if len(candidates) > 1:
        print(f"    [warn] multiple {label} matches for '{stem or suffix_patterns}', using first")
    return candidates[0]

## 4. Core pipeline functions

TIC computation, lognormal fitting and the quality metrics. The MedSAM2 mask for
a frame comes from `Medsam2AdaptiveBboxMasker.compute_frame`, which runs the
coronal + sagittal guide segmentations once per frame, derives a per-z adaptive
bbox from them, and segments each axial slice inside that bbox.

In [11]:
def sam2_mask_for_frame(masker, frame_idx):
    """
    The adaptive-bbox MedSAM2 mask (X, Y, Z) for one CEUS frame.

    `masker.compute_frame` caches its result per frame, and each entry holds a
    full-volume mask *and* the raw volume — roughly 15 MB + 15 MB per frame at
    this data's size. Over a 500-frame TIC that would be tens of GB, so the
    cache is cleared after every frame: the batch loop visits each frame once,
    so there is nothing to reuse.
    """
    mask = masker.compute_frame(frame_idx)["mask_3d"]
    masker.cache.clear()
    return mask

In [12]:
def make_masker(bmode_image_data, seg_data):
    """One adaptive-bbox masker for a case, on the configured checkpoint."""
    return Medsam2AdaptiveBboxMasker(
        seg_data, bmode_image_data, MODEL_CFG, MEDSAM2_CHECKPOINT,
        device=DEVICE,
        smooth_sigma_mm=SMOOTH_SIGMA_MM,
        synthetic_coronal=SYNTHETIC_CORONAL,
    )

In [13]:
# ── TIC + volume helpers ──────────────────────────────────────────────────────
def compute_tic_from_mask(mask_xyz, volume_xyz):
    voxels = volume_xyz[mask_xyz > 0]
    if len(voxels) == 0:
        return np.nan
    return float(np.mean(voxels))

def compute_volume_from_mask(mask_xyz, voxel_volume_mm3):
    if np.sum(mask_xyz) == 0:
        return 0.0
    return float(np.sum(mask_xyz) * voxel_volume_mm3)

def compute_all_tics(image_data, bmode_image_data, seg_data, masker=None,
                     frame_start=None, frame_end=None):
    """Loop over every frame (or a sub-range) and build the raw TICs + a time axis.

    frame_start / frame_end: inclusive/exclusive frame indices (like Python slice).
    Pass None to use the full range.

    `masker` enables the MedSAM2 curve. It is the dominant cost of this function
    by a wide margin -- a full SAM2 image-encoder pass per axial slice, per
    frame, so tens of thousands of forward passes for one case -- which is why
    it was switched off in earlier versions. Pass None to compute only the MC
    and no-MC curves.
    """
    n_frames    = bmode_image_data.pixel_data.shape[-1]
    START_FRAME = int(frame_start) if frame_start is not None else 0
    END_FRAME   = int(frame_end)   if frame_end   is not None else n_frames
    START_FRAME = max(0, START_FRAME)
    END_FRAME   = min(n_frames, END_FRAME)

    tic_sam2, tic_mc, tic_nomc = [], [], []
    vol_sam2, vol_mc, vol_nomc = [], [], []
    frames_computed = []

    voxel_vol = float(np.prod(image_data.pixdim))

    from tqdm import tqdm
    for frame_idx in tqdm(range(START_FRAME, END_FRAME), desc="  TIC frames", leave=False):
        if hasattr(image_data, "intensities_for_analysis"):
            ceus_volume = image_data.intensities_for_analysis[:, :, :, frame_idx]
        else:
            ceus_volume = image_data.pixel_data[:, :, :, frame_idx]

        mc_mask   = seg_data.motion_compensation.apply_to_mask(seg_data.seg_mask, frame_idx, 0)
        nomc_mask = seg_data.seg_mask

        tic_mc.append(compute_tic_from_mask(mc_mask,   ceus_volume))
        tic_nomc.append(compute_tic_from_mask(nomc_mask, ceus_volume))
        vol_mc.append(compute_volume_from_mask(mc_mask,   voxel_vol))
        vol_nomc.append(compute_volume_from_mask(nomc_mask, voxel_vol))

        if masker is not None:
            sam2_mask = sam2_mask_for_frame(masker, frame_idx)
            tic_sam2.append(compute_tic_from_mask(sam2_mask, ceus_volume))
            vol_sam2.append(compute_volume_from_mask(sam2_mask, voxel_vol))

        frames_computed.append(frame_idx)

    frames = np.array(frames_computed)
    if hasattr(image_data, "frame_rate") and image_data.frame_rate > 0:
        time_axis = frames * image_data.frame_rate
        x_label   = "Time (s)"
    else:
        time_axis = frames.astype(float)
        x_label   = "Frame index"

    def decompress(tic):
        return (np.asarray(tic, dtype=float) * 3e4 / 255.0) + 3e4

    out = {
        "frames": frames, "time_axis": time_axis, "x_label": x_label,
        "tic_mc": decompress(tic_mc), "tic_nomc": decompress(tic_nomc),
        "vol_mc": np.array(vol_mc), "vol_nomc": np.array(vol_nomc),
        "has_sam2": masker is not None,
    }
    if masker is not None:
        out["tic_sam2"] = decompress(tic_sam2)
        out["vol_sam2"] = np.array(vol_sam2)
    return out

In [14]:
# ── Lognormal fitting ─────────────────────────────────────────────────────────
def bolus_lognormal(x, auc, mu, sigma, t0):
    with np.errstate(divide="ignore", invalid="ignore"):
        shifted = x - t0
        result = (auc / (shifted * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(-((np.log(shifted) - mu) ** 2) / (2 * sigma ** 2))
        result = np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)
    return result

def fit_lognormal_curve(time, curve):
    """Returns (auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline).
    pe and auc are amplitude above baseline. baseline = amin of input curve."""
    curve = np.array(curve, dtype=float)
    baseline = float(np.amin(curve))
    curve = curve - baseline                   # shift so minimum == 0

    if np.amax(curve) <= 0:
        print("    Curve is constant, cannot normalize.")
        return tuple(np.nan for _ in range(9))
    normalizer = np.amax(curve)
    curve = curve / normalizer                 # normalize to 0-1

    auc_guess   = np.sum(curve) * (time[1] - time[0])
    mu_guess    = np.log(np.argmax(curve) + 1e-8)
    sigma_guess = 0.5
    t0_guess    = time[np.argmax(curve)] * 0.15

    mu_max  = np.log(time[-1]) if time[-1] > 0 else 10.0
    auc_max = (np.sum(curve) * (time[1] - time[0])) * 10.0
    auc_guess = min(auc_guess, auc_max)
    mu_guess  = min(mu_guess,  mu_max)

    try:
        params, _ = curve_fit(
            bolus_lognormal, time, curve,
            p0=(auc_guess, mu_guess, sigma_guess, t0_guess),
            bounds=([0., 0., 0.01, 0.], [auc_max, mu_max, 5.0, time[-1]]),
            method="trf", maxfev=10000)
    except Exception as e:
        print(f"    Error fitting curve: {e}")
        return tuple(np.nan for _ in range(9))

    auc, mu, sigma, t0 = params
    auc = auc * normalizer                     # amplitude above baseline
    mtt = np.exp(mu + sigma**2 / 2)
    tp  = np.exp(mu - sigma**2)

    fitted_curve = bolus_lognormal(time, *params)   # 0-1 normalized space
    pe     = float(np.max(fitted_curve)) * normalizer  # amplitude above baseline
    pe_loc = int(np.argmax(fitted_curve))
    return auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline

def reconstruct_fitted_curve(time, outcome):
    auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline = outcome
    if np.isnan(auc):
        return np.full_like(time, np.nan, dtype=float)
    # auc is in amplitude-above-baseline space; adding baseline gives original units
    return bolus_lognormal(time, auc, mu, sigma, t0) + baseline

def tic_quality_metrics(time, tic_raw, outcome):
    auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline = outcome
    fitted = reconstruct_fitted_curve(time, outcome)   # original TIC units
    valid  = ~np.isnan(fitted)
    ss_res = np.sum((tic_raw[valid] - fitted[valid]) ** 2)
    ss_tot = np.sum((tic_raw[valid] - tic_raw[valid].mean()) ** 2)
    r2     = 1 - ss_res / (ss_tot + 1e-8)
    roughness = np.mean(np.abs(np.diff(tic_raw)))
    baseline_noise = np.std(tic_raw[:5])
    snr = pe / (baseline_noise + 1e-8)         # pe is amplitude above baseline
    return {"r2": r2, "roughness": roughness, "snr": snr,
            "auc": auc, "pe": pe, "tp": tp, "mtt": mtt, "t0": t0}


## 5. Per-case driver

`process_case` ties everything together for one CSV row:
loads the three inputs, computes the TICs, fits each one, writes per-case CSVs +
a comparison plot into `{DEST_ROOT}/{case_key}/`, and returns fit parameters +
quality metrics for **all three methods** (no-MC, MC, MedSAM2) written back to the
master CSV as prefixed columns (`nomc_*`, `mc_*`, `sam2_*`).

> The per-case `*_fit_params.csv` also stores all three methods row-by-row.

In [15]:
def _methods_present(tic_data):
    """Which curves this run actually produced (sam2 only if a masker was used)."""
    return (["sam2"] if tic_data.get("has_sam2") else []) + ["mc", "nomc"]


def _save_case_outputs(case_dir, case_key, tic_data, fits, metrics):
    """Write raw+fitted TIC curve CSV, a fit-params CSV, and a comparison PNG."""
    os.makedirs(case_dir, exist_ok=True)
    t = tic_data["time_axis"]
    methods = _methods_present(tic_data)
    fitted = {m: reconstruct_fitted_curve(t, fits[m]) for m in methods}

    # 1) raw + fitted TIC curves (one row per frame)
    cols = {"frame": tic_data["frames"], "time": t}
    for m in methods:
        cols[f"tic_{m}_raw"] = tic_data[f"tic_{m}"]
        cols[f"tic_{m}_fit"] = fitted[m]
        cols[f"vol_{m}_mm3"] = tic_data[f"vol_{m}"]
    tic_curve_path = os.path.join(case_dir, f"{case_key}_tic_curve.csv")
    pd.DataFrame(cols).to_csv(tic_curve_path, index=False)

    # 2) fit params + quality, one row per method
    plabels = ["auc", "pe", "tp", "mtt", "t0", "mu", "sigma", "pe_loc", "baseline"]
    rows = []
    for method in methods:
        outcome = fits[method]
        m = metrics[method]
        row = {"method": method}
        for lab, val in zip(plabels, outcome):
            row[lab] = val
        row["r2"]        = m["r2"]
        row["roughness"] = m["roughness"]
        row["snr"]       = m["snr"]
        # Each method's own segmented volume, taken at its first non-empty frame.
        # Previously every row carried the no-MC volume, which made the MedSAM2
        # and MC rows describe a volume neither of them actually segmented.
        vol = tic_data[f"vol_{method}"]
        vol_pos = vol[vol > 0]
        row["volume_mm3"] = float(vol_pos[0]) if vol_pos.size else 0.0
        rows.append(row)
    fit_params_path = os.path.join(case_dir, f"{case_key}_fit_params.csv")
    pd.DataFrame(rows).to_csv(fit_params_path, index=False)

    # 3) comparison plot
    style = {"sam2": ("blue", "darkblue", "MedSAM2"),
             "mc":   ("red", "darkred", "MC"),
             "nomc": ("green", "darkgreen", "No MC")}
    fig, ax = plt.subplots(figsize=(10, 5))
    for m in methods:
        raw_c, fit_c, label = style[m]
        ax.plot(t, tic_data[f"tic_{m}"], color=raw_c, lw=2, alpha=0.4, label=f"{label} raw")
        ax.plot(t, fitted[m], color=fit_c, lw=2, ls="--", label=f"{label} fit")
    ax.set_xlabel(tic_data["x_label"]); ax.set_ylabel("Mean intensity")
    ax.set_title(case_key); ax.legend(loc="upper right"); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = os.path.join(case_dir, f"{case_key}_tic_plot.png")
    fig.savefig(plot_path, dpi=120); plt.close(fig)

    return {"tic_curve": tic_curve_path, "fit_params": fit_params_path, "plot": plot_path}

In [16]:
def process_case(row, dest_root=DEST_ROOT, frame_start=None, frame_end=None,
                 run_sam2=True):
    """Full pipeline for a single CSV row. Returns a dict of results + status.

    frame_start / frame_end: restrict TIC computation to this frame window
    (inclusive start, exclusive end). Set to None to use all frames.
    run_sam2: False computes only the MC / no-MC curves (fast).
    """
    case_key = make_case_key(row["Site"], row["Patient Number"], row["Visit"], row["Bolus"])
    print(f"\n=== {case_key} ===")

    inputs = locate_inputs(row)
    if not all([inputs["bmode"], inputs["ceus"], inputs["seg"]]):
        missing = [k for k in ("bmode", "ceus", "seg") if not inputs[k]]
        print(f"  [SKIP] missing inputs: {missing}")
        return {"case_key": case_key, "status": f"missing:{','.join(missing)}"}

    print(f"  bmode: {os.path.basename(inputs['bmode'])}")
    print(f"  ceus : {os.path.basename(inputs['ceus'])}")
    print(f"  seg  : {os.path.basename(inputs['seg'])}")

    # ── Load data (same entrypoints as single-case notebook) ──────────────────
    image_data       = scan_loading_step(SCAN_TYPE, inputs["ceus"])
    bmode_image_data = scan_loading_step(SCAN_TYPE, inputs["bmode"])
    seg_data         = seg_loading_step(SEG_TYPE, image_data, inputs["seg"], inputs["ceus"])

    # ── MedSAM2 masker (one per case, freed below) ────────────────────────────
    masker = make_masker(bmode_image_data, seg_data) if run_sam2 else None
    if masker is not None:
        bbox = seg_data.motion_compensation.tracked_bboxes[0]
        n_z = int(bbox.z_max) - int(bbox.z_min)
        n_f = (frame_end or bmode_image_data.pixel_data.shape[-1]) - (frame_start or 0)
        print(f"  MedSAM2: ~{n_f} frames x ~{n_z} axial slices "
              f"= ~{n_f * (n_z + 2):,} SAM2 forward passes")

    started = time.time()
    try:
        tic_data = compute_all_tics(image_data, bmode_image_data, seg_data,
                                    masker=masker,
                                    frame_start=frame_start, frame_end=frame_end)
    finally:
        del masker
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    print(f"  TICs computed in {time.time() - started:.0f}s")

    t = tic_data["time_axis"]
    methods = _methods_present(tic_data)
    fits    = {m: fit_lognormal_curve(t, tic_data[f"tic_{m}"]) for m in methods}
    metrics = {m: tic_quality_metrics(t, tic_data[f"tic_{m}"], fits[m]) for m in methods}

    # destination: prefer the row's save-path column if present, else DEST_ROOT
    row_dest = _clean(row.get("TIC curve save path", "")) or dest_root
    case_dir = os.path.join(row_dest, case_key)
    paths = _save_case_outputs(case_dir, case_key, tic_data, fits, metrics)
    print(f"  wrote: {case_dir}")

    def _method_results(prefix, fit_out, met, vol_arr):
        auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline = fit_out
        vol_pos = vol_arr[vol_arr > 0]
        vol_mm3 = float(vol_pos[0]) if vol_pos.size else 0.0
        return {
            f"{prefix}_AUC": auc, f"{prefix}_PE": pe, f"{prefix}_TP": tp,
            f"{prefix}_MTT": mtt, f"{prefix}_T0": t0, f"{prefix}_Mu": mu,
            f"{prefix}_Sigma": sigma, f"{prefix}_R2": met["r2"],
            f"{prefix}_roughness": met["roughness"], f"{prefix}_snr": met["snr"],
            f"{prefix}_volume_mm3": vol_mm3,
        }

    result = {"case_key": case_key, "status": "ok",
              "case_dir": case_dir, "paths": paths}
    for m in methods:
        result.update(_method_results(m, fits[m], metrics[m], tic_data[f"vol_{m}"]))
    return result

## 6. Batch run

Loads the master CSV, runs each selected case, and writes the fitted parameters
and quality metrics back into the matching row. The CSV is saved after **every**
case, so a crash mid-batch keeps the completed work.

⚠️ **Budget the time.** The MedSAM2 curve needs one SAM2 forward pass per axial
slice per frame — printed per case before it starts. A 500-frame acquisition
with a 60-slice lesion is ~30k passes for that case alone, so a full 41-row
batch is an overnight job. Narrow it with `ONLY_CASES`, or with the frame window
in section 8, before committing to the whole CSV.

Existing per-case files and master-CSV rows are overwritten in place
(`OVERWRITE = True`).

In [17]:
# Load master CSV
df = pd.read_csv(MASTER_CSV)
# normalise the Site column (strip stray trailing spaces seen in the file)
df["Site"] = df["Site"].astype(str).str.strip()
print(f"Loaded {len(df)} rows from master CSV")

PARAM_COLS = ["AUC", "PE", "TP", "MTT", "T0", "Mu", "Sigma",
              "R2", "roughness", "snr", "volume_mm3"]
RESULT_COLS = [f"{prefix}_{c}" for prefix in ("sam2", "mc", "nomc") for c in PARAM_COLS]

for col in RESULT_COLS:
    if col not in df.columns:
        df[col] = np.nan

# Keep a timestamped backup before overwriting a CSV that already holds results.
if OVERWRITE and df[RESULT_COLS].notna().any().any():
    backup = MASTER_CSV.replace(".csv", f"_backup_{time.strftime('%Y%m%d_%H%M%S')}.csv")
    pd.read_csv(MASTER_CSV).to_csv(backup, index=False)
    print(f"backed up existing results -> {backup}")

print(f"result columns: {len(RESULT_COLS)} ({', '.join(RESULT_COLS[:3])}, ...)")

Loaded 41 rows from master CSV
backed up existing results -> /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/Motion Compensation Comparison 4 patients(MedSAM2)_backup_20260915_131552.csv
result columns: 33 (sam2_AUC, sam2_PE, sam2_TP, ...)


In [ ]:
summary = []

for idx, row in df.iterrows():
    case_key = make_case_key(row["Site"], row["Patient Number"], row["Visit"], row["Bolus"])

    if ONLY_CASES is not None and case_key not in ONLY_CASES:
        continue

    # OVERWRITE is the point of this run; only skip finished rows when resuming.
    if not OVERWRITE and pd.notna(row.get("sam2_R2", np.nan)):
        print(f"[skip-done] {case_key}")
        summary.append({"case_key": case_key, "status": "skipped-done"})
        continue

    try:
        result = process_case(row, run_sam2=True)
    except Exception as e:
        print(f"  [ERROR] {case_key}: {e}")
        traceback.print_exc()
        result = {"case_key": case_key, "status": f"error:{e}"}

    summary.append(result)

    # write results back into the matching row & persist immediately
    if result.get("status") == "ok":
        for col in RESULT_COLS:
            if col in result:
                df.at[idx, col] = result[col]
        df.to_csv(MASTER_CSV, index=False)
        print(f"  master CSV updated for {case_key}")

print("\nBatch complete.")

## 7. Run summary

In [ ]:
summary_df = pd.DataFrame(summary)
if not summary_df.empty:
    print(summary_df["status"].value_counts().to_string())
summary_df

## 8. (Optional) Re-run a single case with frame-range selection

Use the config cell below to pick the case (by CSV row index or case key) and
optionally restrict the TIC analysis to a **good-frame window** — useful when
the acquisition contains bad frames (motion artefacts, dropout, etc.).

| Variable | Meaning |
|---|---|
| `RERUN_ROW_IDX` | Integer index into the master CSV (`df.iloc[N]`). |
| `FRAME_START` | First frame to include (0-based, inclusive). `None` = start of acquisition. |
| `FRAME_END` | First frame to **exclude** (exclusive). `None` = end of acquisition. |

Results are saved to the same per-case folder and the master CSV is updated.

In [ ]:
df

In [18]:
# ── Frame-range selection ────────────────────────────────────────────────────
# Row to re-run (0-based index into the master CSV)
RERUN_ROW_IDX = 39

# Restrict TIC analysis to frames [FRAME_START, FRAME_END).
# Set either to None to use the full acquisition boundary.
# Example: skip the first 3 bad frames and stop at frame 90:
#   FRAME_START = 3
#   FRAME_END   = 90
FRAME_START = 0   # e.g. 3
FRAME_END   = 510   # e.g. 90

# ── Load CSV and preview the selected row ────────────────────────────────────
df = pd.read_csv(MASTER_CSV)
row = df.iloc[RERUN_ROW_IDX]

n_frames = None  # will be filled after data load below
print(f"Selected row {RERUN_ROW_IDX}:", make_case_key(
    row["Site"], row["Patient Number"], row["Visit"], row["Bolus"]))
if FRAME_START is not None or FRAME_END is not None:
    print(f"  Frame window: [{FRAME_START}, {FRAME_END})")
else:
    print("  Frame window: full acquisition")
row

Selected row 39: SHC-P09-V01-CE2
  Frame window: [0, 510)


Site                                                                 SHC
Patient Number                                                       P09
Visit                                                                V01
Bolus                                                                CE2
Data Dir               /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...
TIC curve save path    /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...
nomc_AUC                                                   697250.985925
nomc_PE                                                        4924.4596
nomc_TP                                                        25.014647
nomc_MTT                                                      186.114612
nomc_T0                                                         6.796147
nomc_Mu                                                         4.557396
nomc_Sigma                                                      1.156691
nomc_R2                                            

In [19]:
# Re-run the selected case with the chosen frame window
res = process_case(row, frame_start=FRAME_START, frame_end=FRAME_END)
res


=== SHC-P09-V01-CE2 ===
  bmode: SHC-P09-V01-CE2_18.45.33_mf_sip_capture_50_2_1_0_BMODE.nii
  ceus : SHC-P09-V01-CE2_18.45.33_mf_sip_capture_50_2_1_0_CEUS.nii
  seg  : SHC-P09-V01-CE2-MC_VOI.nii.gz
  MedSAM2: ~510 frames x ~174 axial slices = ~89,760 SAM2 forward passes


  TICs computed in 736s
  wrote: /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2


{'case_key': 'SHC-P09-V01-CE2',
 'status': 'ok',
 'case_dir': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2',
 'paths': {'tic_curve': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2/SHC-P09-V01-CE2_tic_curve.csv',
  'fit_params': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2/SHC-P09-V01-CE2_fit_params.csv',
  'plot': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2/SHC-P09-V01-CE2_tic_plot.png'},
 'sam2_AUC': np.float64(1020540.0228118089),
 'sam2_PE': np.float64(6332.47541276028),
 'sam2_TP': np.float64(24.52604884078283),
 'sam2_MTT': np.float64(237.34744390218594),
 'sam2_T0': np.float64(6.496155224963014),
 'sam2_Mu': np.float64(4.712928640655661),
 'sam2_Sigma': np.float64(1.2301190470559258),
 'sam2_R2': np.float64(0.7913486177410024),
 'sam2_roughness': np.float64(561.5924589261981),
 'sam2_snr': np